# NBA Pipeline Smoke Test

Loads curated Parquet via DuckDB views — no live `nba_api` calls.

Run a refresh first:
```bash
cd nba_pipeline
python -m nba_pipeline refresh --all
```

In [ ]:
from pathlib import Path
import sys

# Allow importing the package when the notebook cwd is nba_pipeline/notebooks
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nba_pipeline.config import Settings
from nba_pipeline.load.duckdb_views import connect

cfg = Settings(project_root=ROOT)
con = connect(cfg)
print("Curated root:", cfg.curated_dir)

In [ ]:
print(con.execute("SHOW TABLES").fetchdf())
print("players:", con.execute("SELECT COUNT(*) FROM players").fetchone()[0])
print("teams:", con.execute("SELECT COUNT(*) FROM teams").fetchone()[0])
print("player_season_stats:", con.execute("SELECT COUNT(*) FROM player_season_stats").fetchone()[0])
print("shot_charts:", con.execute("SELECT COUNT(*) FROM shot_charts").fetchone()[0])

In [ ]:
# Archetype clustering inputs: per-100 stats, minutes filter
stats = con.execute("""
SELECT season, PLAYER_NAME, MIN, PTS, AST, REB, STL, BLK, TOV, FG3A, FTA
FROM player_season_stats
WHERE MIN >= 500
ORDER BY season, PTS DESC
LIMIT 10
""").fetchdf()
stats

In [ ]:
# Shot chart sample (e.g. high-volume shooters)
shots = con.execute("""
SELECT season, PLAYER_NAME, SHOT_MADE_FLAG, SHOT_DISTANCE, LOC_X, LOC_Y
FROM shot_charts
WHERE PLAYER_NAME ILIKE '%Curry%'
LIMIT 20
""").fetchdf()
shots